# Занятие 9. Семинар: градиентный бустинг

План занятия:

1. смещение и разброс, простые и глубокие деревья, почему бэггинг не уменьшает смещение;
2. стекинг и блендинг;
3. бустинг на остатках вручную;
4. градиентный бустинг: остатки как антиградиент, другие функции потерь;
5. темп обучения, число и глубина деревьев;
6. смещение и разброс бустинга;
7. библиотеки: гистограммный бустинг, XGBoost (extreme gradient boosting), LightGBM
   (light gradient boosting machine) и CatBoost (categorical boosting);
8. итоговое сравнение моделей;
9. вопросы: прогноз вне диапазона ответов, бустинг линейных моделей, что параллелится.

Код, который только рисует, вынесен в файл `sem09_utils.py` рядом с ноутбуком: в ячейках
остаются обучение моделей, эксперименты и вызовы рисующих функций.

Подробные выводы, бинарная и многоклассовая классификация, AdaBoost как бустинг
с экспоненциальной функцией потерь, переобучение и ранняя остановка, стохастический бустинг
и устройство библиотек — в конспекте `notes_09_boosting.ipynb`.

In [ ]:
import time
import warnings

import numpy as np
import pandas as pd

from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from scipy.special import expit
from sklearn.datasets import make_friedman1, make_hastie_10_2, make_moons
from sklearn.ensemble import (
    BaggingClassifier, BaggingRegressor, GradientBoostingClassifier, GradientBoostingRegressor,
    HistGradientBoostingRegressor, RandomForestClassifier, RandomForestRegressor,
)
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_val_predict, train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

from sem09_utils import (
    plot_bars, plot_bias_variance_curve, plot_bias_variance_targets, plot_boosting_progress,
    plot_boosting_steps, plot_ensemble_scheme, plot_fit_panels, plot_fits, plot_gallery, plot_staged_curves,
    plot_tree_depths, show_table,
)

warnings.filterwarnings("ignore", category=UserWarning)
SEED = 42

## 1. Смещение простой модели

На занятии 8 ошибку модели раскладывали на шум, смещение и разброс.
Бэггинг усредняет $B$ моделей, обученных на бутстрэп-выборках:
$a(x) = \frac{1}{B}\sum_{k=1}^{B} b_k(x)$. Среднее предсказание ансамбля совпадает со средним
предсказанием одной модели, $\mathbb{E}\,a(x) = \mathbb{E}\,b_1(x)$, поэтому смещение у них одинаковое.
Разброс ансамбля равен $\rho\sigma^2 + \frac{1 - \rho}{B}\sigma^2$, где $\sigma^2$ — разброс одной модели, а $\rho$ — коэффициент
корреляции предсказаний двух моделей ансамбля; с ростом $B$ он уменьшается до $\rho\sigma^2$. Бэггинг выгоден для моделей с малым смещением —
глубоких деревьев; смещение слишком простой модели он не уменьшает.

Данные для рисунков — синус с шумом, как на занятии 8: $y = \sin x + \varepsilon$,
$\varepsilon \sim \mathcal{N}(0,\, 0.25^2)$, $x$ равномерно на $[0, 2\pi]$; 80 точек обучающей выборки и 2000 точек
тестового сета из того же распределения.

In [ ]:
rng_sine = np.random.default_rng(0)

x_sine = np.sort(rng_sine.uniform(0, 2 * np.pi, 80))[:, None]        # (80, 1): обучающая выборка
noise_sine = rng_sine.normal(0, 0.25, 80)
y_sine = np.sin(x_sine[:, 0]) + noise_sine

x_sine_test = rng_sine.uniform(0, 2 * np.pi, 2000)[:, None]          # (2000, 1): тестовый сет
noise_sine_test = rng_sine.normal(0, 0.25, 2000)
y_sine_test = np.sin(x_sine_test[:, 0]) + noise_sine_test

x_grid = np.linspace(0, 2 * np.pi, 400)[:, None]                      # (400, 1): точки для рисунков

print("обучение:", x_sine.shape, " тест:", x_sine_test.shape)

Разложение ошибки удобно представить мишенью. Центр мишени — истинное значение $f(x)$
в фиксированной точке $x$, каждая синяя точка — предсказание в этой точке модели,
обученной на одной из случайных обучающих выборок. **Смещение** — насколько центр облака
точек удален от центра мишени, **разброс** — насколько точки рассеяны вокруг своего центра.

In [ ]:
plot_bias_variance_targets()

Под каждой мишенью — модели, которые обычно в нее попадают. Линейная регрессия на синусе
и пень — левый нижний угол: слишком простая модель на любой выборке дает почти одно
и то же предсказание, но мимо цели. Глубокое дерево и kNN с $k = 1$ — правый верхний:
в среднем попадают в цель, но каждая модель следует за шумом своей выборки. Линейная
регрессия по многим шумным признакам на малой выборке — правый нижний: форма зависимости
не та, и коэффициенты подстраиваются под шум. Бэггинг сжимает облако к его центру
и уменьшает разброс; бустинг сдвигает облако к центру мишени и уменьшает смещение;
оба ведут в левый верхний угол.

**Пень** (decision stump) — дерево глубины 1: один вопрос «$x \le c$?» и два листа,
в каждом листе — среднее ответов попавших в него объектов. Чем больше глубина,
тем больше вопросов и листьев: у дерева глубины $d$ не больше $2^d$ листьев.
**Глубокое дерево** — дерево без ограничения глубины: вопросы задают, пока в листе
не останется объектов с одинаковым ответом, то есть обычно по одному объекту.

Сверху — устройство трех деревьев, обученных на синусе: вопросы в узлах, значения
в листьях (оранжевые — положительные, синие — отрицательные); у глубокого дерева
листьев так много, что показаны только узлы и ребра. Снизу — их предсказания;
пунктир — пороги вопросов. MSE (mean squared error, среднеквадратичная ошибка)
на тестовом сете — $\frac{1}{m}\sum_i (y_i - a(x_i))^2$ по его объектам.

In [ ]:
depth_cases = {"пень": 1, "дерево глубины 3": 3, "глубокое дерево: глубина не ограничена": None}
depth_models = {}
depth_test_mse = {}

for name, max_depth in depth_cases.items():
    model = DecisionTreeRegressor(max_depth=max_depth, random_state=SEED)
    model.fit(x_sine, y_sine)

    test_prediction = model.predict(x_sine_test)
    depth_models[name] = model
    depth_test_mse[name] = np.mean((test_prediction - y_sine_test) ** 2)

plot_tree_depths(depth_models, depth_test_mse, x_sine, y_sine, x_grid)

Пень делит ось одним порогом $x \le 3.50$ на две части и предсказывает на каждой
константу: синус он не повторяет, MSE на тестовом сете 0.176. Дерево глубины 3 задает
7 вопросов и делит ось на 8 участков: ступенчатая кривая близка к синусу, MSE 0.087.
Глубокое дерево задает 79 вопросов, у него 80 листьев — по одному на объект обучающей
выборки, и предсказание проходит через каждую точку вместе с шумом: MSE 0.125.
Пень недообучен — большое смещение; глубокое дерево переобучено — большой разброс.

Бэггинг из 100 пней и бэггинг из 100 глубоких деревьев на этой выборке:

In [ ]:
stump = DecisionTreeRegressor(max_depth=1, random_state=SEED)
stump.fit(x_sine, y_sine)

bagged_stumps = BaggingRegressor(DecisionTreeRegressor(max_depth=1), n_estimators=100, random_state=SEED)
bagged_stumps.fit(x_sine, y_sine)

deep_tree = DecisionTreeRegressor(random_state=SEED)
deep_tree.fit(x_sine, y_sine)

bagged_deep_trees = BaggingRegressor(DecisionTreeRegressor(), n_estimators=100, random_state=SEED)
bagged_deep_trees.fit(x_sine, y_sine)

bagging_models = {
    "пень": stump,
    "бэггинг 100 пней": bagged_stumps,
    "глубокое дерево": deep_tree,
    "бэггинг 100 глубоких деревьев": bagged_deep_trees,
}
bagging_test_mse = {}
for name, model in bagging_models.items():
    test_prediction = model.predict(x_sine_test)
    bagging_test_mse[name] = np.mean((test_prediction - y_sine_test) ** 2)

bagging_panels = []
for title, names in [("простая модель: смещение остается", ["пень", "бэггинг 100 пней"]),
                     ("сложная модель: разброс уменьшается", ["глубокое дерево", "бэггинг 100 глубоких деревьев"])]:
    curves = {f"{name}: MSE на тесте {bagging_test_mse[name]:.3f}": bagging_models[name].predict(x_grid) for name in names}
    bagging_panels.append((title, curves))

plot_fit_panels(bagging_panels, x_sine, y_sine, x_grid)

Слева: пень 0.176, бэггинг из 100 пней 0.148. Пни, обученные на разных бутстрэп-выборках,
ставят порог в разных местах между $x \approx 2.9$ и 3.6, и бэггинг сглаживает ступеньку
на этом участке. На остальной оси все пни предсказывают почти одну и ту же константу,
и синус не повторяет ни один из них: среднее одинаково смещенных моделей смещено так же.

Справа: глубокое дерево 0.125, бэггинг из 100 глубоких деревьев 0.092 — ошибка меньше
на четверть. Каждое дерево проходит через шум своей бутстрэп-выборки; шум у разных деревьев разный
и при усреднении частично гасится, а форма синуса у всех деревьев общая и остается. Ниже
0.0625 — дисперсии шума — не опустится ни одна модель. Бэггинг уменьшает разброс и не
трогает смещение, поэтому в бэггинг и случайный лес берут глубокие деревья. Оценка смещения
и разброса по 300 выборкам — в разделе 6.

Чтобы уменьшить смещение, модели строят **последовательно**: каждая следующая
модель исправляет ошибки суммы предыдущих. Такой ансамбль называют
**бустингом** (boosting). Схема бэггинга, бустинга и стекинга — ансамбля из моделей разных
типов, который разбирают в разделе 2:

In [ ]:
plot_ensemble_scheme()

В бэггинге деревья не знают друг о друге, и порядок их обучения не важен. В стекинге
модели разных типов обучают независимо, а объединяет их не среднее, а метамодель.
В бустинге каждое следующее дерево обучают на том, что не объяснила сумма предыдущих,
поэтому деревья строят строго по очереди.

## 2. Стекинг и блендинг

**Стекинг** (stacking, stacked generalization; Wolpert, 1992) объединяет модели разных
типов $a_1, \dots, a_M$ и подбирает способ объединения по данным: ответы базовых моделей
становятся признаками **метамодели** $g$,

$$a(x) = g\big(a_1(x), \dots, a_M(x)\big).$$

Обычно $g$ — линейная модель, и ответ — взвешенная сумма $a(x) = w_0 + \sum_{m} w_m\, a_m(x)$
с весами, подобранными по данным. Бэггинг — частный случай с одинаковыми моделями
и равными весами $w_m = \frac{1}{M}$.

Базовые модели на синусе — линейная регрессия, kNN с $k = 10$ и глубокое дерево;
MSE каждой на тестовом сете:

In [ ]:
def make_base_models():
    """Три базовые модели разных типов."""
    models = {
        "линейная регрессия": LinearRegression(),
        "kNN, k = 10": KNeighborsRegressor(n_neighbors=10),
        "глубокое дерево": DecisionTreeRegressor(random_state=SEED),
    }
    return models


base_models = make_base_models()
stacking_results = {}
train_predictions = []        # предсказания базовых моделей на объектах, на которых они обучены
test_predictions = []

for name, model in base_models.items():
    model.fit(x_sine, y_sine)
    train_predictions.append(model.predict(x_sine))
    test_predictions.append(model.predict(x_sine_test))
    stacking_results[name] = np.mean((test_predictions[-1] - y_sine_test) ** 2)

train_predictions = np.column_stack(train_predictions)      # (80, 3): метапризнаки обучающей выборки
test_predictions = np.column_stack(test_predictions)        # (2000, 3)
print(pd.Series(stacking_results).round(4))

Метамодель обучают на ответах базовых моделей. Если взять ответы на тех же объектах,
на которых базовые модели обучены, глубокое дерево не ошибается ни на одном из них:
оно повторяет каждую точку обучающей выборки. Метамодель видит идеальный признак
и отдает весь вес ему:

In [ ]:
naive_meta = LinearRegression()
naive_meta.fit(train_predictions, y_sine)

naive_prediction = naive_meta.predict(test_predictions)
stacking_results["метамодель на предсказаниях обучения"] = np.mean((naive_prediction - y_sine_test) ** 2)

print("веса метамодели:", dict(zip(base_models, naive_meta.coef_.round(3))))

Веса метамодели: 0 у линейной регрессии и kNN и 1 у глубокого дерева; MSE на тестовом
сете 0.126 — как у одного глубокого дерева. Это утечка: на обучающей выборке глубокое
дерево выглядит идеальным, а на новых объектах ошибается больше kNN. Ответы модели
на объектах, на которых она обучена, не показывают, как она ошибается на новых.

**Стекинг.** Метапризнаки строят по объектам, которых базовая модель не видела, как
в кросс-валидации: выборку делят на $K$ блоков, каждую базовую модель обучают на $K - 1$
блоках и предсказывают оставшийся. После $K$ повторов у каждого объекта есть
предсказание вне блока (out-of-fold) — от модели, обученной без него. Метамодель
обучают на этих предсказаниях, а базовые модели для новых объектов обучают на всей
выборке. В sklearn — `StackingRegressor` и `StackingClassifier` с параметром `cv`.

In [ ]:
folds = KFold(n_splits=5, shuffle=True, random_state=SEED)

out_of_fold = []
for name, model in make_base_models().items():
    predictions = cross_val_predict(model, x_sine, y_sine, cv=folds)     # каждый объект предсказан моделью без него
    out_of_fold.append(predictions)
out_of_fold = np.column_stack(out_of_fold)                                # (80, 3)

stacking_meta = LinearRegression()
stacking_meta.fit(out_of_fold, y_sine)

stacking_prediction = stacking_meta.predict(test_predictions)            # базовые модели обучены на всей выборке
stacking_results["стекинг, 5 блоков"] = np.mean((stacking_prediction - y_sine_test) ** 2)

print("веса метамодели:", dict(zip(base_models, stacking_meta.coef_.round(3))))

**Блендинг** (blending) — упрощенный стекинг: выборку один раз делят на две части,
базовые модели обучают на первой, метамодель — на их предсказаниях на второй. Базовые
модели обучают один раз, но и они, и метамодель видят только часть данных.

In [ ]:
x_part_base, x_part_meta, y_part_base, y_part_meta = train_test_split(x_sine, y_sine, test_size=0.3, random_state=SEED)

blend_meta_features = []
blend_test_features = []
for name, model in make_base_models().items():
    model.fit(x_part_base, y_part_base)                          # 70% выборки — базовым моделям
    blend_meta_features.append(model.predict(x_part_meta))       # 30% — метамодели
    blend_test_features.append(model.predict(x_sine_test))

blending_meta = LinearRegression()
blending_meta.fit(np.column_stack(blend_meta_features), y_part_meta)

blending_prediction = blending_meta.predict(np.column_stack(blend_test_features))
stacking_results["блендинг, 70% / 30%"] = np.mean((blending_prediction - y_sine_test) ** 2)

print("веса метамодели:", dict(zip(base_models, blending_meta.coef_.round(3))))
show_table(pd.DataFrame({"MSE на тесте": stacking_results}), digits=4)

Со стекингом метамодель дает kNN вес 0.77, глубокому дереву 0.27, линейной регрессии −0.10;
MSE на тестовом сете 0.074 — меньше, чем у лучшей базовой модели (kNN, 0.075), и меньше,
чем у глубокого дерева (0.126). Блендинг дает 0.088: базовые модели обучены на 56 объектах
из 80, метамодель — на 24, и обе части слабее. Стекинг выигрывает, когда базовые модели
ошибаются по-разному.

## 3. Бустинг на остатках

Первый широко применяемый бустинг — **AdaBoost** (adaptive boosting; Freund, Schapire, 1997)
для бинарной классификации. Следующему пню он передает не остатки, а веса объектов: объекты,
на которых ансамбль ошибается, весят больше. AdaBoost — бустинг с экспоненциальной функцией
потерь; алгоритм, вывод весов и пример на плоскости — в конспекте, раздел 6, реализация —
задача 4 лабораторной. Семинар разбирает бустинг на остатках и его обобщение на любую
функцию потерь — градиентный бустинг.

Модель бустинга — сумма начального приближения и $T$ деревьев:

$$a_T(x) = a_0 + b_1(x) + b_2(x) + \dots + b_T(x).$$

Начальное приближение $a_0$ — константа, для квадратичной ошибки — среднее ответов $\bar{y}$.
Дерево $b_t$ обучают не на ответах $y_i$, а на **остатках** (residuals) текущей модели

$$r_{ti} = y_i - a_{t-1}(x_i)$$

и добавляют к модели: $a_t(x) = a_{t-1}(x) + b_t(x)$. На рисунке остаток — вертикальный отрезок
от кривой модели до точки: насколько модель промахнулась на объекте и в какую
сторону. Если бы дерево повторило остатки точно, $b_t(x_i) = r_{ti}$, новая модель совпала бы
с ответами: $a_t(x_i) = a_{t-1}(x_i) + y_i - a_{t-1}(x_i) = y_i$. Дерево ограниченной глубины повторяет остатки
приближенно, и следующее дерево исправляет то, что осталось.

Базовая модель в этом разделе — пень: каждый шаг добавляет к модели одну ступеньку.
Первый шаг вручную. Качество модели на обучающей выборке измеряют той же MSE —
средним квадратом остатков $\frac{1}{n}\sum_i r_i^2$.

In [ ]:
initial_value = y_sine.mean()                                  # a_0 = ȳ
prediction_0 = np.full(len(y_sine), initial_value)             # a_0(x_i) для всех объектов обучения

residuals_1 = y_sine - prediction_0                            # r_1i = y_i − a_0

stump_1 = DecisionTreeRegressor(max_depth=1, random_state=SEED)
stump_1.fit(x_sine, residuals_1)

prediction_1 = prediction_0 + stump_1.predict(x_sine)          # a_1(x_i) = a_0 + b_1(x_i)

threshold_1 = stump_1.tree_.threshold[0]
leaf_values_1 = stump_1.tree_.value[[1, 2], 0, 0]              # значения листов «да» и «нет»

mse_0 = np.mean((y_sine - prediction_0) ** 2)
mse_1 = np.mean((y_sine - prediction_1) ** 2)

print(f"a_0 = {initial_value:.3f}")
print(f"пень b_1: x ≤ {threshold_1:.2f}? да: {leaf_values_1[0]:+.3f}, нет: {leaf_values_1[1]:+.3f}")
print(f"MSE на обучении: a_0 {mse_0:.4f}, a_1 {mse_1:.4f}")

Те же шаги в цикле для 100 пней. На каждом шаге сохраняют пень, остатки
и предсказания модели на обучающей выборке, на тестовом сете и в точках рисунка.

In [ ]:
n_steps = 100

initial_value = y_sine.mean()
current_train = np.full(len(y_sine), initial_value)          # a_{t-1}(x_i) на обучающей выборке
current_test = np.full(len(y_sine_test), initial_value)      # a_{t-1}(x) на тестовом сете
current_grid = np.full(len(x_grid), initial_value)           # a_{t-1}(x) в точках рисунка

stumps = []                      # b_1, ..., b_T
residuals_by_step = []           # r_t: остатки, на которых обучен пень b_t
models_on_train = [current_train]   # a_0, a_1, ..., a_T на обучающей выборке
models_on_grid = [current_grid]     # a_0, a_1, ..., a_T в точках рисунка
train_mse = [np.mean((y_sine - current_train) ** 2)]
test_mse = [np.mean((y_sine_test - current_test) ** 2)]

for step in range(n_steps):
    residuals = y_sine - current_train

    stump = DecisionTreeRegressor(max_depth=1, random_state=SEED)
    stump.fit(x_sine, residuals)

    current_train = current_train + stump.predict(x_sine)
    current_test = current_test + stump.predict(x_sine_test)
    current_grid = current_grid + stump.predict(x_grid)

    stumps.append(stump)
    residuals_by_step.append(residuals)
    models_on_train.append(current_train)
    models_on_grid.append(current_grid)
    train_mse.append(np.mean((y_sine - current_train) ** 2))
    test_mse.append(np.mean((y_sine_test - current_test) ** 2))

best_test_step = int(np.argmin(test_mse))
for T in [0, 1, 3, 10, 100]:
    print(f"T = {T:3d}: MSE на обучении {train_mse[T]:.4f}, на тесте {test_mse[T]:.4f}")
print(f"наименьшая MSE на тесте {test_mse[best_test_step]:.4f} при T = {best_test_step}")

Первые четыре шага, по столбцу на шаг. Сверху — модель до шага и ее остатки
(оранжевые отрезки), снизу — те же остатки точками и пень, обученный на них, с его
вопросом и значениями листьев; в заголовке — MSE на обучении до и после шага. Шкала
по вертикали у всех восьми графиков общая.

In [ ]:
plot_boosting_steps(stumps, residuals_by_step, models_on_train, models_on_grid, train_mse, x_sine, y_sine, x_grid)

Первый пень задает вопрос «$x \le 3.50$?»: к левой части оси прибавляет 0.58, к правой −0.65,
и MSE на обучающей выборке падает с 0.504 до 0.126. Остатки второго шага по модулю
меньше; на правом краю $x > 5.78$ модель занижает ответы, и второй пень поднимает этот
участок на 0.36. Третий пень поднимает участок $x \le 2.45$ на 0.17, четвертый опускает левый
край $x \le 0.45$ на 0.52. Каждый пень исправляет самое крупное отклонение, оставшееся после
предыдущих; после четырех шагов MSE на обучении 0.066.

Слева — первые шесть пней по отдельности: каждый — одна ступенька. В середине — их
сумма вместе с $a_0$ после 1, 6 и 100 пней. Справа — MSE на обучающей выборке и на тестовом
сете в зависимости от числа пней (по вертикали логарифмическая шкала).

In [ ]:
plot_boosting_progress(stumps, models_on_grid, train_mse, test_mse, x_sine, y_sine, x_grid, noise_variance=0.25 ** 2)

Каждый пень — ступенька с одним порогом, а сумма многих ступенек — кривая с многими
порогами, близкая к синусу. Пень, обученный на остатках, не увеличивает ошибку
на обучающей выборке: значение листа — среднее остатков его объектов, а сумма квадратов
отклонений от среднего не больше суммы квадратов самих остатков, $\sum_{i}(r_i - \bar{r})^2 \le \sum_{i} r_i^2$.
Поэтому MSE на обучении не возрастает и к 100 пням падает до 0.022 — ниже дисперсии
шума 0.0625: модель начинает повторять шум обучающей выборки. MSE на тестовом сете
наименьшая при $T = 56$ (0.086) и к $T = 100$ растет до 0.090: с ростом числа деревьев бустинг
переобучается. Средства против этого — темп обучения, ограничение глубины и ранняя
остановка, раздел 5 и конспект, раздел 7.

В sklearn тот же алгоритм — `GradientBoostingRegressor`. Параметр
`learning_rate=1.0` означает, что деревья складываются без множителя
(раздел 5).

In [ ]:
library_model = GradientBoostingRegressor(n_estimators=n_steps, learning_rate=1.0, max_depth=1, random_state=SEED)
library_model.fit(x_sine, y_sine)

library_train = library_model.predict(x_sine)
library_grid = library_model.predict(x_grid)

difference_train = np.max(np.abs(library_train - current_train))
difference_grid = np.max(np.abs(library_grid - models_on_grid[-1]))

print(f"начальное приближение sklearn {library_model.init_.constant_[0, 0]:.6f}, среднее y {initial_value:.6f}")
print(f"наибольшее расхождение с ручным циклом: обучение {difference_train:.1e}, точки рисунка {difference_grid:.1e}")

Расхождение на уровне ошибок округления: ручной цикл и sklearn строят одну
и ту же модель.

## 4. Градиентный бустинг

Модель после $t - 1$ шагов — сумма начального приближения и деревьев:

$$a_{t-1}(x) = a_0 + \sum_{s=1}^{t-1} b_s(x).$$

На шаге $t$ ищут дерево $b(x; \theta)$ с параметрами $\theta$ — вопросами узлов и значениями листьев —
и множитель $\rho$:

$$(\rho_t, \theta_t) = \arg\min_{\rho,\, \theta} \sum_{i=1}^{n} L\big(y_i,\ a_{t-1}(x_i) + \rho\, b(x_i; \theta)\big).$$

Для произвольной функции потерь $L$ эту задачу в явном виде не решить. **Градиентный бустинг**
(gradient boosting; Friedman, 2001) заменяет ее тремя простыми:

$$r_{ti} = -\left[\frac{\partial L(y_i, a)}{\partial a}\right]_{a = a_{t-1}(x_i)}, \qquad i = 1, \dots, n;$$

$$\theta_t = \arg\min_{\theta} \sum_{i=1}^{n} \big(r_{ti} - b(x_i; \theta)\big)^2;$$

$$\rho_t = \arg\min_{\rho} \sum_{i=1}^{n} L\big(y_i,\ a_{t-1}(x_i) + \rho\, b(x_i; \theta_t)\big), \qquad a_t(x) = a_{t-1}(x) + \eta\, \rho_t\, b(x; \theta_t).$$

**Псевдоостатки** (pseudo-residuals) $r_{ti}$ — антиградиент потери по предсказанию: в какую сторону
сдвинуть предсказание на объекте $i$, чтобы потеря убывала быстрее всего. Дерево, обученное по MSE
на парах $(x_i, r_{ti})$, задает направление шага в любой точке $x$, множитель $\rho_t$ — длину шага
по исходной $L$, $\eta$ — темп обучения (раздел 5).

Для MSE, $L = \frac{1}{2}(y - a)^2$, производная $\partial L/\partial a = -(y - a)$: псевдоостаток — обычный остаток
$y_i - a_{t-1}(x_i)$, $\rho_t = 1$, и алгоритм совпадает с бустингом на остатках из раздела 3. В sklearn
вместо одного $\rho_t$ на дерево подбирают свой сдвиг в каждом листе по исходной потере. Выводы,
схема шага и шаги бустинга в пространстве прогнозов — в конспекте, раздел 2.

Другие функции потерь регрессии — **средняя абсолютная ошибка** (mean absolute error, MAE),
$L = |y - a|$, с псевдоостатком $\operatorname{sign}(y - a)$ и медианой остатков в листе, и **функция Хьюбера**
(Huber loss), квадратичная при $|y - a| \le \delta$ и линейная дальше, с псевдоостатком, обрезанным
по модулю на уровне $\delta$. Выводы и таблица функций потерь — в конспекте, раздел 3.

Шесть ответов обучающей выборки сдвигают вверх на 4 — выбросы:

In [ ]:
rng_outliers = np.random.default_rng(3)
outlier_index = rng_outliers.choice(len(y_sine), 6, replace=False)

y_outliers = y_sine.copy()
y_outliers[outlier_index] = y_outliers[outlier_index] + 4          # шесть ответов сдвинуты вверх на 4

print("номера выбросов:", np.sort(outlier_index))

Бустинг с тремя функциями потерь на тех же данных, 300 пней, $\eta = 0.1$. Качество — MSE
на тестовом сете, в котором выбросов нет: насколько модель описывает саму зависимость,
а не выбросы.

In [ ]:
robust_losses = {"squared_error": "MSE", "absolute_error": "MAE", "huber": "функцией Хьюбера"}
robust_curves = {}

for loss, label in robust_losses.items():
    model = GradientBoostingRegressor(loss=loss, n_estimators=300, learning_rate=0.1, max_depth=1, random_state=SEED)
    model.fit(x_sine, y_outliers)

    robust_prediction = model.predict(x_sine_test)
    robust_test_mse = np.mean((robust_prediction - y_sine_test) ** 2)
    robust_curves[f"бустинг с {label}: MSE на тесте {robust_test_mse:.3f}"] = model.predict(x_grid)

# для сравнения: тот же бустинг с MSE на данных без выбросов
clean_model = GradientBoostingRegressor(n_estimators=300, learning_rate=0.1, max_depth=1, random_state=SEED)
clean_model.fit(x_sine, y_sine)
clean_prediction = clean_model.predict(x_sine_test)
clean_test_mse = np.mean((clean_prediction - y_sine_test) ** 2)
print(f"MSE на тесте бустинга с MSE на данных без выбросов: {clean_test_mse:.3f}")

plot_fits(robust_curves, x_sine, y_outliers, x_grid, highlight=outlier_index, legend_outside=True, size=(11, 5))

MSE на тестовом сете: 0.540 с MSE, 0.101 с MAE и 0.095 с функцией Хьюбера. Тот же бустинг
с MSE на данных без выбросов дает 0.082, а ниже дисперсии шума 0.0625 не опустится ни одна
модель. Шесть выбросов из 80 увеличили ошибку бустинга с MSE в шесть с лишним раз, а с MAE
и функцией Хьюбера ошибка всего на 16–23% выше, чем у бустинга на данных без выбросов. С MSE псевдоостаток выброса близок к 4, разбиения пней выделяют участки
с выбросами, и модель поднимается к выбросам вместе с соседними точками.
С MAE псевдоостаток выброса равен $+1$, как у любой точки выше модели, а значение
листа — медиана остатков, на которую несколько выбросов почти не влияют.
Функция Хьюбера ограничивает псевдоостатки выбросов уровнем $\delta$, а на остальных
точках совпадает с MSE. Устойчивость зависит и от глубины: дерево глубины 2
может выделить выброс в отдельный лист. Те же три модели с деревьями глубины 2:

In [ ]:
for loss, label in robust_losses.items():
    model = GradientBoostingRegressor(loss=loss, n_estimators=300, learning_rate=0.1, max_depth=2, random_state=SEED)
    model.fit(x_sine, y_outliers)

    robust_prediction = model.predict(x_sine_test)
    robust_test_mse = np.mean((robust_prediction - y_sine_test) ** 2)
    print(f"бустинг с {label}, глубина 2: MSE на тесте {robust_test_mse:.3f}")

MSE на тесте с деревьями глубины 2: 2.004 с MSE, 1.570 с функцией Хьюбера и 0.091
с MAE. Дерево глубины 2 выделяет выброс в отдельный лист, и значение такого листа
у MSE и у функции Хьюбера — сдвиг к выбросу: у функции Хьюбера ограничен псевдоостаток,
по которому строят дерево, а значение листа считают по остаткам. За 300 деревьев модель
доходит до выбросов. У MAE разбиение по знакам остатков, выделяющее один выброс, дает
малый прирост, и такие листья почти не появляются.

## 5. Темп обучения, число и глубина деревьев

Далее используются данные Фридмана из занятия 8: 10 признаков, равномерных на $[0, 1]$,

$$y = 10\sin(\pi x_1 x_2) + 20(x_3 - 0.5)^2 + 10x_4 + 5x_5 + \varepsilon, \qquad \varepsilon \sim \mathcal{N}(0, 1);$$

от признаков $x_6, \dots, x_{10}$ ответ не зависит. 500 объектов обучающей выборки и 5000
тестового сета. Функция `evaluate_regressor` обучает модель и записывает
MSE на тестовом сете и время обучения в общую таблицу `friedman_results`;
все модели семинара сравнивают через нее. Функция `staged_mse` вычисляет
MSE после каждого дерева: метод `staged_predict` выдает предсказания модели
из первых 1, 2, …, $T$ деревьев по очереди.

In [ ]:
Xf, yf = make_friedman1(n_samples=5500, noise=1.0, random_state=0)

Xf_train, yf_train = Xf[:500], yf[:500]           # 500 объектов обучения
Xf_test, yf_test = Xf[500:], yf[500:]             # 5000 объектов тестового сета

friedman_results = {}


def evaluate_regressor(name, model):
    """Обучить модель на данных Фридмана и записать в таблицу MSE на тестовом сете и время обучения."""
    started = time.perf_counter()
    model.fit(Xf_train, yf_train)
    fit_time = time.perf_counter() - started

    prediction = model.predict(Xf_test)
    test_mse = np.mean((prediction - yf_test) ** 2)

    friedman_results[name] = {"MSE на тесте": test_mse, "время обучения, с": fit_time}
    print(f"{name}: MSE на тесте {test_mse:.3f}, обучение {fit_time:.2f} с")
    return model


def staged_mse(model, X, y):
    """MSE после 1, 2, ..., T деревьев: staged_predict выдает предсказания по очереди."""
    errors = []
    for prediction in model.staged_predict(X):
        errors.append(np.mean((prediction - y) ** 2))

    errors = np.array(errors)
    return errors


print("обучение:", Xf_train.shape, " тест:", Xf_test.shape)

Для сравнения обучают линейную регрессию, одно дерево, случайный лес и бустинг
с параметрами sklearn по умолчанию: 100 деревьев глубины 3, $\eta = 0.1$.

In [ ]:
evaluate_regressor("линейная регрессия", LinearRegression())
evaluate_regressor("дерево, глубина 6", DecisionTreeRegressor(max_depth=6, random_state=SEED))
evaluate_regressor("случайный лес, 300 деревьев", RandomForestRegressor(n_estimators=300, random_state=SEED, n_jobs=-1))
default_boosting = evaluate_regressor("бустинг, 100 деревьев глубины 3", GradientBoostingRegressor(random_state=SEED))

show_table(pd.DataFrame(friedman_results).T)

MSE на тестовом сете: 7.11 у линейной регрессии, 9.20 у дерева глубины 6, 4.45
у случайного леса и 2.81 у бустинга с параметрами по умолчанию — в 1.6 раза
меньше, чем у леса. Нижняя граница MSE любой модели — дисперсия шума
$\sigma_\varepsilon^2 = 1$.

**Темп обучения** (learning rate, shrinkage) $\eta$ уменьшает вклад каждого дерева:
$a_t(x) = a_{t-1}(x) + \eta\, b_t(x)$. При $\eta = 1$ дерево исправляет остатки целиком, и уже
через несколько шагов остатки состоят в основном из шума, под который
подгоняются следующие деревья. При малом $\eta$ дерево исправляет долю $\eta$
ошибки, следующие деревья обучаются на близких остатках и строят близкие поправки;
итоговая поправка — сумма многих деревьев, и случайные особенности отдельного
дерева в ней усредняются. Однако при меньшем $\eta$ нужно больше деревьев. Кривые MSE на тестовом сете для четырех значений $\eta$, до 2000 деревьев
глубины 3; точка — минимум кривой.

In [ ]:
rates = [1.0, 0.3, 0.1, 0.03]
max_trees = 2000
rate_curves = {}

for rate in rates:
    model = GradientBoostingRegressor(n_estimators=max_trees, learning_rate=rate, max_depth=3, random_state=SEED)
    model.fit(Xf_train, yf_train)
    rate_curves[rate] = staged_mse(model, Xf_test, yf_test)

rate_rows = []
for rate, curve in rate_curves.items():
    best_index = int(np.argmin(curve))
    rate_rows.append({
        "темп η": str(rate),
        "лучшее число деревьев": best_index + 1,
        "MSE при лучшем числе": curve[best_index],
        f"MSE при {max_trees} деревьях": curve[-1],
    })
rate_table = pd.DataFrame(rate_rows).set_index("темп η")

plot_staged_curves({f"η = {rate}": curve for rate, curve in rate_curves.items()},
                   "число деревьев T", "MSE на тестовом сете", ylim=(0, 12), noise_level=1.0)
show_table(rate_table)

При $\eta = 1$ MSE на тестовом сете наименьшая при 9 деревьях (6.63) и затем растет
до 7.41: каждое дерево подгоняется под шум остатков. При $\eta = 0.3$ минимум 3.29
при 151 дереве, при $\eta = 0.1$ — 2.60 при 438, при $\eta = 0.03$ — 2.53 при 836. Чем меньше $\eta$,
тем ниже минимум и тем больше деревьев до него. Заметно ошибка растет после минимума
только при $\eta = 1$, с 6.63 до 7.41; при $\eta \le 0.3$ ошибка 2000 деревьев отличается
от минимума меньше чем на 0.03. Поэтому на практике фиксируют небольшое $\eta$, от 0.03 до 0.1,
и подбирают число деревьев.

**Глубина деревьев.** На пути от корня до листа дерево глубины $d$ задает
не больше $d$ вопросов, поэтому описывает взаимодействие не больше $d$ признаков.
Пень задает один вопрос, и каждый пень — функция одного признака $x_{j(t)}$.
Сумму пней группируют по признакам:

$$a_T(x) = a_0 + \sum_{t=1}^{T} \eta\, b_t(x_{j(t)}) = a_0 + \sum_{j=1}^{p} g_j(x_j), \qquad g_j(x_j) = \sum_{t:\, j(t) = j} \eta\, b_t(x_j).$$

Бустинг пней — **аддитивная модель**: сумма функций отдельных признаков.
Слагаемое $10\sin(\pi x_1 x_2)$ зависит от произведения двух признаков и в такую
сумму не раскладывается. Кривые для глубин от 1 до 6 при $\eta = 0.1$:

In [ ]:
depths = [1, 2, 3, 4, 6]
depth_curves = {}

for depth in depths:
    model = GradientBoostingRegressor(n_estimators=1000, learning_rate=0.1, max_depth=depth, random_state=SEED)
    model.fit(Xf_train, yf_train)
    depth_curves[depth] = staged_mse(model, Xf_test, yf_test)

depth_rows = []
for depth, curve in depth_curves.items():
    best_index = int(np.argmin(curve))
    depth_rows.append({"глубина": depth, "лучшее число деревьев": best_index + 1, "MSE при лучшем числе": curve[best_index]})
depth_table = pd.DataFrame(depth_rows).set_index("глубина")

plot_staged_curves({f"глубина {depth}": curve for depth, curve in depth_curves.items()},
                   "число деревьев T, темп η = 0.1", "MSE на тестовом сете", ylim=(0, 12), noise_level=1.0)
show_table(depth_table)

У пней наименьшая MSE 3.46: аддитивная модель не описывает взаимодействие
$x_1 x_2$. Деревья глубины 2 и 3 его описывают, и ошибка падает до 2.65 и 2.60.
При глубине 4 и 6 она снова растет, до 2.72 и 3.56: каждое дерево сложнее, чем
нужно, и подгоняется под шум. Поэтому в бустинге используют неглубокие деревья:
Фридман рекомендовал деревья с 4–8 листьями, то есть глубины 2–3, в XGBoost глубина
по умолчанию 6. В лесу деревья растят без ограничения глубины.

## 6. Смещение и разброс бустинга

Слагаемые ошибки оценивают на синусе, как на занятии 8: 300 обучающих выборок
по 80 точек. Для пня и бэггинга из 100 пней — функция `simulate_bias_variance`
занятия 8; для бустинга пней с $\eta = 0.1$ — то же на тех же выборках после 1, 3, 10, …,
300 деревьев.

In [ ]:
def simulate_bias_variance(make_model, n_datasets=300, n_train=80, noise=0.25, seed=0):
    """Смещение², разброс и ошибка модели на синусе по многим обучающим выборкам."""
    rng = np.random.default_rng(seed)

    x_eval = np.linspace(0, 2 * np.pi, 200)[:, None]      # (200, 1): точки, где оценивают слагаемые
    f_eval = np.sin(x_eval[:, 0])                         # (200,): истинная функция в этих точках

    predictions = np.empty((n_datasets, len(x_eval)))     # строка b — предсказания модели, обученной на выборке b
    for b in range(n_datasets):
        x_sample = rng.uniform(0, 2 * np.pi, n_train)[:, None]
        noise_sample = rng.normal(0, noise, n_train)
        y_sample = np.sin(x_sample[:, 0]) + noise_sample

        model = make_model()
        model.fit(x_sample, y_sample)
        predictions[b] = model.predict(x_eval)

    mean_prediction = predictions.mean(axis=0)            # (200,): среднее предсказание по выборкам
    bias_squared = np.mean((mean_prediction - f_eval) ** 2)
    variance = np.mean(predictions.var(axis=0))

    y_new = f_eval + rng.normal(0, noise, predictions.shape)
    direct_error = np.mean((y_new - predictions) ** 2)

    estimate = {
        "смещение²": bias_squared,
        "разброс": variance,
        "дисперсия шума": noise ** 2,
        "сумма трех слагаемых": bias_squared + variance + noise ** 2,
        "ошибка напрямую": direct_error,
    }
    result = (estimate, x_eval, predictions)
    return result


def simulate_boosting_stages(stages, learning_rate=0.1, n_datasets=300, n_train=80, noise=0.25, seed=0):
    """Смещение², разброс и ошибка бустинга пней после stages[k] деревьев; выборки те же, что в simulate_bias_variance."""
    rng = np.random.default_rng(seed)

    x_eval = np.linspace(0, 2 * np.pi, 200)[:, None]
    f_eval = np.sin(x_eval[:, 0])

    predictions = np.empty((len(stages), n_datasets, len(x_eval)))     # [число деревьев, выборка, точка]
    for b in range(n_datasets):
        x_sample = rng.uniform(0, 2 * np.pi, n_train)[:, None]
        noise_sample = rng.normal(0, noise, n_train)
        y_sample = np.sin(x_sample[:, 0]) + noise_sample

        model = GradientBoostingRegressor(n_estimators=max(stages), learning_rate=learning_rate, max_depth=1, random_state=SEED)
        model.fit(x_sample, y_sample)

        position = 0
        for number, stage_prediction in enumerate(model.staged_predict(x_eval), start=1):
            if number == stages[position]:
                predictions[position, b] = stage_prediction
                position = position + 1
                if position == len(stages):
                    break

    y_new = f_eval + rng.normal(0, noise, (n_datasets, len(x_eval)))
    rows = []
    for position, n_trees in enumerate(stages):
        stage_predictions = predictions[position]
        mean_prediction = stage_predictions.mean(axis=0)

        bias_squared = np.mean((mean_prediction - f_eval) ** 2)
        variance = np.mean(stage_predictions.var(axis=0))
        direct_error = np.mean((y_new - stage_predictions) ** 2)

        rows.append({
            "модель": f"бустинг пней, T = {n_trees}",
            "смещение²": bias_squared,
            "разброс": variance,
            "дисперсия шума": noise ** 2,
            "сумма трех слагаемых": bias_squared + variance + noise ** 2,
            "ошибка напрямую": direct_error,
        })

    table = pd.DataFrame(rows).set_index("модель")
    return table


def make_stump():
    model = DecisionTreeRegressor(max_depth=1, random_state=SEED)
    return model


def make_bagged_stumps():
    model = BaggingRegressor(DecisionTreeRegressor(max_depth=1), n_estimators=100, random_state=SEED)
    return model


stump_estimate, _, _ = simulate_bias_variance(make_stump)
bagging_estimate, _, _ = simulate_bias_variance(make_bagged_stumps)

boosting_stages = [1, 3, 10, 30, 100, 200, 300]
boosting_table = simulate_boosting_stages(boosting_stages)

reference_rows = pd.DataFrame([stump_estimate, bagging_estimate], index=["пень", "бэггинг, 100 пней"])
boosting_bias_variance = pd.concat([reference_rows, boosting_table])
show_table(boosting_bias_variance, digits=4)

Слагаемые бустинга в зависимости от числа пней; пунктир того же цвета —
бэггинг из 100 пней.

In [ ]:
plot_bias_variance_curve(boosting_stages, boosting_table, bagging_estimate, noise_variance=0.25 ** 2)

У пня смещение² 0.075 и разброс 0.031. Бэггинг из 100 пней уменьшает разброс
вдвое, до 0.015, а смещение² почти не меняет: 0.073. У бустинга пней с $\eta = 0.1$
смещение² падает с 0.420 при одном пне (при $\eta = 0.1$ он сдвигает начальную константу
на десятую долю своих значений) до 0.0018 при 200 пнях, а разброс растет с 0.007 до 0.016:
каждый следующий пень подгоняется под остатки конкретной выборки. Сумма трех
слагаемых наименьшая при 200 пнях — 0.080; на занятии 8 лучшее одно дерево
(глубина 4) давало 0.096, бэггинг деревьев без ограничения глубины — 0.095.

Бэггинг уменьшает разброс и не меняет смещение, бустинг уменьшает смещение
ценой роста разброса. Поэтому в бэггинге и лесу используют глубокие деревья
с малым смещением, а в бустинге — неглубокие с малым разбросом.

## 7. Библиотеки

`GradientBoostingRegressor` в каждом узле перебирает все пороги всех признаков,
как дерево занятия 8. Специализированные библиотеки ускоряют этот перебор
и добавляют регуляризацию.

**Гистограммный бустинг.** Значения каждого признака один раз до обучения
разбивают по квантилям не больше чем на 255 интервалов (бинов) и заменяют
номером интервала. В узле за один проход по его объектам для каждого бина
накапливают суммы $G$ и $H$ первых и вторых производных функции потерь, затем
перебирают не больше 254 границ между бинами вместо всех порогов. Гистограмму одного потомка
получают вычитанием гистограммы другого из гистограммы родителя. В sklearn —
`HistGradientBoostingRegressor` и `HistGradientBoostingClassifier`; они обрабатывают
пропуски и категориальные признаки, а при числе объектов больше 10 000
по умолчанию включают раннюю остановку.

**XGBoost** (extreme gradient boosting; Chen, Guestrin, 2016) использует первую
и вторую производные любой функции потерь, как шаг Ньютона для бинарной классификации (конспект, раздел 4),
и добавляет к сумме потерь штраф за сложность дерева $\mu J + \frac{\lambda}{2}\sum_{j=1}^{J} w_j^2$,
где $J$ — число листьев, $w_j$ — значения листьев (в XGBoost параметры `gamma` и `lambda`).
Для листа с суммами $G$ и $H$ разложение потери до второго порядка, как в конспекте, раздел 4,
вместе со штрафом дает функцию $G w + \frac{1}{2}(H + \lambda)\,w^2$; ее производная $G + (H + \lambda)\,w$
обращается в ноль в точке $w = -\frac{G}{H + \lambda}$. Значение листа и прирост разбиения узла
на потомков $l$ и $r$:

$$w_j = -\frac{G_j}{H_j + \lambda}, \qquad \text{Gain} = \frac{1}{2}\Big(\frac{G_l^2}{H_l + \lambda} + \frac{G_r^2}{H_r + \lambda} - \frac{(G_l + G_r)^2}{H_l + H_r + \lambda}\Big) - \mu.$$

При $\lambda = 0$ значение листа — шаг Ньютона из конспекта, раздел 4; $\lambda > 0$ уменьшает по модулю
значения всех листьев, и сильнее — листьев с малой суммой $H$ (для MSE $h_i = 1$, и $H$ —
число объектов листа). Разбиение с $\text{Gain} < 0$ не делают: штраф $\mu$ уже вычтен.
Вывод формулы прироста — в конспекте.

**LightGBM** (light gradient boosting machine; Ke и соавторы, 2017):

- гистограммы, как выше;
- рост дерева по листьям (leaf-wise): каждый раз делят лист с наибольшим приростом,
  как sklearn при `max_leaf_nodes` на занятии 8; размер дерева задает `num_leaves`;
- GOSS (gradient-based one-side sampling, односторонняя выборка по градиенту):
  объекты с наибольшими $|g_i|$ оставляют все, а из остальных берут случайную часть
  с увеличенным весом;
- EFB (exclusive feature bundling, объединение взаимоисключающих признаков):
  разреженные признаки, которые не бывают ненулевыми одновременно, объединяют
  в один.

**CatBoost** (categorical boosting; Прохоренкова и соавторы, 2018):

- симметричные деревья (oblivious trees): во всех узлах одного уровня один и тот же
  вопрос, поэтому дерево глубины $d$ — таблица из $2^d$ значений, и предсказание
  вычисляется быстро;
- упорядоченный бустинг (ordered boosting, `boosting_type="Ordered"`): псевдоостаток
  объекта вычисляют моделью, при обучении которой этот объект не использовался;
  в примерах ниже CatBoost работает в обычном режиме `Plain`;
- категориальные признаки кодируют средним ответом по объектам, стоящим раньше
  в случайной перестановке (ordered target statistics), — без утечки ответа,
  которую разбирали на занятии 4.

Четыре реализации с одинаковыми настройками — 300 деревьев глубины 3, $\eta = 0.1$ —
на 500 объектах данных Фридмана, в общей таблице:

In [ ]:
def make_library_models(n_trees=300):
    """Четыре реализации градиентного бустинга с одинаковыми настройками: n_trees деревьев глубины 3, η = 0.1."""
    models = {
        "GradientBoostingRegressor": GradientBoostingRegressor(
            n_estimators=n_trees, learning_rate=0.1, max_depth=3, random_state=SEED),
        "HistGradientBoostingRegressor": HistGradientBoostingRegressor(
            max_iter=n_trees, learning_rate=0.1, max_depth=3, early_stopping=False, random_state=SEED),
        "LightGBM": LGBMRegressor(
            n_estimators=n_trees, learning_rate=0.1, max_depth=3, num_leaves=8, random_state=SEED, verbose=-1),
        "CatBoost": CatBoostRegressor(
            iterations=n_trees, learning_rate=0.1, depth=3, random_seed=SEED, verbose=0, allow_writing_files=False),
    }
    return models


for name, model in make_library_models().items():
    evaluate_regressor(name, model)

MSE на тестовом сете: 2.62 у `GradientBoostingRegressor`, 2.36
у `HistGradientBoostingRegressor`, 2.30 у LightGBM и 1.85 у CatBoost. Число
и глубина деревьев одинаковые, но остальные параметры по умолчанию различаются:
`HistGradientBoostingRegressor` и LightGBM требуют не меньше 20 объектов в листе,
CatBoost прибавляет к знаменателю значения листа $\lambda = 3$ и строит симметричные
деревья, которые меньше подгоняются под отдельные объекты. На 500 объектах
такая регуляризация помогает; на других данных порядок может быть другим.

Разница во времени заметна на больших выборках. Те же четыре модели
на 10 000 объектах обучения (тестовый сет — 5000 объектов из того же распределения):

In [ ]:
X_big, y_big = make_friedman1(n_samples=15000, noise=1.0, random_state=1)
X_big_train, y_big_train = X_big[:10000], y_big[:10000]
X_big_test, y_big_test = X_big[10000:], y_big[10000:]

timing_rows = {}
for name, model in make_library_models().items():
    started = time.perf_counter()
    model.fit(X_big_train, y_big_train)
    fit_time = time.perf_counter() - started

    prediction = model.predict(X_big_test)
    test_mse = np.mean((prediction - y_big_test) ** 2)
    timing_rows[name] = {"MSE на тесте": test_mse, "время обучения, с": fit_time}

timing_table = pd.DataFrame(timing_rows).T
show_table(timing_table)

На 10 000 объектах MSE четырех реализаций близки, от 1.07 до 1.18, а время
обучения различается в десятки раз: `GradientBoostingRegressor` обучается
несколько секунд, CatBoost — доли секунды (точные числа зависят от компьютера).
Гистограммы делают поиск разбиения независимым от числа различных значений
признака, а симметричное дерево CatBoost строится по уровням: на каждом уровне
выбирают один вопрос сразу для всех узлов.

## 8. Итоговое сравнение

Все модели занятий 8 и 9 на двух лунах (300 точек, 40% — тестовый сет, как на занятии 8).
Сверху — устройство модели: одно дерево целиком или первые три дерева ансамбля,
снизу — вероятность класса 1 на плоскости; точки с белой обводкой — обучающая выборка,
с черной — тестовый сет; число в углу — accuracy на тестовом сете.

In [ ]:
X_gallery, y_gallery = make_moons(n_samples=300, noise=0.3, random_state=0)
Xg_train, Xg_test, yg_train, yg_test = train_test_split(X_gallery, y_gallery, test_size=0.4, random_state=SEED)

gallery_models = [
    ("пень", DecisionTreeClassifier(max_depth=1, random_state=SEED), "одно дерево"),
    ("дерево глубины 3", DecisionTreeClassifier(max_depth=3, random_state=SEED), "одно дерево"),
    ("глубокое дерево", DecisionTreeClassifier(random_state=SEED), "одно дерево"),
    ("бэггинг: 100 глубоких деревьев", BaggingClassifier(DecisionTreeClassifier(), n_estimators=100, random_state=SEED),
     "среднее вероятностей 100 деревьев на бутстрэп-выборках"),
    ("случайный лес: 100 деревьев", RandomForestClassifier(n_estimators=100, random_state=SEED),
     "то же + в каждом узле случайный признак из двух"),
    ("бустинг: 100 деревьев глубины 2",
     GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=2, random_state=SEED),
     "сумма оценок 100 деревьев, обученных по очереди"),
]

gallery_entries = []
gallery_scores = {}
for name, model, how in gallery_models:
    model.fit(Xg_train, yg_train)
    test_accuracy = model.score(Xg_test, yg_test)
    gallery_scores[name] = test_accuracy

    if how == "одно дерево":
        members = None
    elif isinstance(model, GradientBoostingClassifier):
        members = list(model.estimators_[:, 0])
    else:
        members = list(model.estimators_)
    gallery_entries.append({"name": name, "model": model, "members": members, "how": how, "accuracy": test_accuracy})

plot_gallery(gallery_entries, Xg_train, yg_train, Xg_test, yg_test)
show_table(pd.DataFrame({"accuracy на тесте": gallery_scores}))

Пень делит плоскость одной прямой, accuracy 0.78; дерево глубины 3 — 0.86; глубокое дерево —
0.91, но его граница рваная, с узкими полосами вокруг отдельных точек. Бэггинг и лес
усредняют по 100 глубоких деревьев (у первых трех от 11 до 23 листьев) и дают 0.96: ступени
границы мельче, разброс меньше. На двух признаках лес в каждом узле выбирает один признак
из двух, и здесь его accuracy совпала с бэггингом. Бустинг складывает 100 маленьких деревьев
по 4 листа и дает 0.92. Параметры моделей не подбирали: на данных Фридмана бустинг лучше
леса, а на лунах с параметрами по умолчанию — хуже; какая модель лучше, зависит от данных
и подбора параметров.

Все модели семинара на данных Фридмана в одной таблице и на одном графике.

In [ ]:
final_table = pd.DataFrame(friedman_results).T.sort_values("MSE на тесте")

plot_bars(final_table["MSE на тесте"].to_dict(), "MSE на тестовом сете", "", horizontal=True,
          reference=1.0, reference_label=r"дисперсия шума $\sigma_\varepsilon^2 = 1$")
show_table(final_table)

Все варианты бустинга, от 1.85 до 2.81, ошибаются меньше случайного леса (4.45),
линейной регрессии (7.11) и одного дерева (9.20). На занятии 8 ошибка леса
на этих данных состояла в основном из смещения, и бустинг уменьшает именно его.


## Вопросы

**Может ли случайный лес предсказать значение больше наибольшего ответа обучающей выборки?**
Нет. Значение листа — среднее ответов его объектов, прогноз леса — среднее значений листьев,
а среднее не выходит за пределы $[\min_i y_i,\ \max_i y_i]$. Бустинг — может: прогноз
$a_0 + \eta \sum_t \gamma_t$ — сумма поправок, и диапазон ответов ее не ограничивает. Проверка:
$y = x_1 + x_2$ без шума, в обучающей выборке только объекты с $x_1 + x_2 \le 1.2$; в точке $(1, 1)$
истинный ответ равен 2:

In [ ]:
rng_range = np.random.default_rng(5)
X_square = rng_range.uniform(0, 1, size=(4000, 2))
X_range = X_square[X_square.sum(axis=1) <= 1.2][:600]      # только объекты с x₁ + x₂ ≤ 1.2
y_range = X_range.sum(axis=1)                               # y = x₁ + x₂ без шума

forest_range = RandomForestRegressor(n_estimators=200, random_state=SEED)
forest_range.fit(X_range, y_range)

boosting_range = GradientBoostingRegressor(n_estimators=300, learning_rate=0.1, max_depth=1, random_state=SEED)
boosting_range.fit(X_range, y_range)

corner = np.array([[1.0, 1.0]])
forest_corner = forest_range.predict(corner)[0]
boosting_corner = boosting_range.predict(corner)[0]
print(f"наибольший ответ обучающей выборки: {y_range.max():.3f}")
print(f"в точке (1, 1), где y = 2: случайный лес {forest_corner:.3f}, бустинг пней {boosting_corner:.3f}")

Лес не выходит за наибольший ответ обучения, 1.20. Бустинг пней — аддитивная модель
$f_1(x_1) + f_2(x_2)$ (раздел 5): по каждому признаку в отдельности он видел значения до 1
и в точке $(1, 1)$ складывает обе поправки — 1.92 при истинных 2.

**Зачем бустить линейные модели?** Незачем: сумма линейных функций — линейная функция,

$$\sum_{t=1}^{T} \big(c_t + w_t^\top x\big) = \Big(\sum_{t=1}^{T} c_t\Big) + \Big(\sum_{t=1}^{T} w_t\Big)^{\top} x,$$

и бустинг линейных регрессий по MSE приходит к той же линейной регрессии, что и метод
наименьших квадратов, только за много шагов. Бустингу нужны модели, сумма которых
выходит за пределы их семейства, — деревья.

**Что параллелится?** В лесе деревья обучают независимо и строят параллельно (`n_jobs`).
В бустинге деревья строят строго по очереди: следующему дереву нужны псевдоостатки суммы
предыдущих. Параллелят построение одного дерева — поиск лучшего разбиения по признакам
и по частям выборки; так устроены XGBoost, LightGBM и CatBoost, в том числе на видеокарте.

**Что будет, если обучать деревья на градиенте, а не на антиградиенте?** Каждый шаг пойдет
в сторону роста потерь. Для MSE и дерева, которое повторяет остатки точно, $b_t(x_i) = -r_{ti}$, и

$$r_{t+1, i} = y_i - a_t(x_i) = y_i - a_{t-1}(x_i) + \eta\, r_{ti} = (1 + \eta)\, r_{ti}:$$

остатки растут в $1 + \eta$ раз за шаг, MSE на обучении — в $(1 + \eta)^2$ раз.


## Итоги

- бэггинг уменьшает разброс и не меняет смещение; бустинг строит модели
  последовательно и уменьшает смещение;
- в случайном лесе — сложные модели, глубокие деревья. Усреднение уменьшает только
  разброс, а смещение почти не меняет: у бэггинга 100 пней смещение² 0.073 против 0.075
  у одного пня, разброс 0.015 против 0.031 (раздел 6): среднее пней остается пнем
  с меньшим разбросом. Поэтому в лес берут модели с малым смещением и большим разбросом:
  глубокое дерево повторяет обучающую выборку и в среднем попадает в цель, а его
  разброс гасит усреднение. Случайный выбор признаков в разбиениях делает деревья
  менее похожими друг на друга, и усреднение гасит больше разброса. На полумесяцах
  глубокое дерево дает accuracy 0.908, лес из 100 таких деревьев — 0.958 (раздел 8);
- в бустинге — простые модели: пни и неглубокие деревья. Смещение уменьшает сумма:
  у бустинга пней смещение² падает с 0.420 при $T = 1$ до 0.001 при $T = 300$, а разброс
  растет медленно, с 0.007 до 0.017 (раздел 6). От базовой модели нужен малый разброс
  и шаг примерно в сторону антиградиента; ее смещение исправят следующие деревья, как
  пни в пространстве прогнозов (конспект, раздел 2). Глубокое дерево подстраивается под шум
  обучающей выборки, и следующие деревья учат остатки, в которых все больше шума;
  дерево без ограничения глубины повторяет выборку, остатки после него равны нулю,
  и бустинг заканчивается на первом, переобученном дереве. На данных Фридмана
  при глубине 6 лучшая MSE 3.56 против 2.60 при глубине 3. Глубина — наибольший
  порядок взаимодействия признаков, поэтому слишком мелкие деревья тоже проигрывают,
  если в зависимости есть взаимодействия: у пней на тех же данных MSE 3.46 (раздел 5).
  По умолчанию глубина 3 в `GradientBoostingRegressor` и 6 в XGBoost и CatBoost,
  а в LightGBM и `HistGradientBoostingRegressor` ограничивают число листьев, 31;
- стекинг: ответы базовых моделей разных типов объединяет метамодель, обученная
  на их предсказаниях вне блоков кросс-валидации; на предсказаниях обучающей выборки
  метамодель отдает весь вес переобученной модели; блендинг делит выборку один раз;
- бустинг на остатках: каждое дерево обучают на $y_i - a_{t-1}(x_i)$ и прибавляют
  к модели;
- остаток — антиградиент квадратичной ошибки по предсказанию; градиентный
  бустинг обучает дерево на антиградиенте любой дифференцируемой функции
  потерь и пересчитывает значения листьев по исходной функции потерь;
- MAE и функция Хьюбера ограничивают псевдоостатки выбросов и делают модель
  устойчивой к ним;
- темп обучения $\eta$ и число деревьев $T$ связаны: меньшее $\eta$ требует больше деревьев
  и обычно дает меньшую ошибку;
- глубина дерева — наибольший порядок взаимодействия признаков; бустинг пней —
  аддитивная модель;
- в отличие от леса, бустинг переобучается с ростом $T$; число деревьев выбирают
  ранней остановкой по валидационной выборке (конспект, раздел 7);
- XGBoost и LightGBM выбирают разбиения по первым и вторым производным
  с регуляризацией значений листьев $\lambda$; гистограммные реализации
  (`HistGradientBoostingRegressor`, LightGBM, CatBoost) на 10 000 объектах обучаются
  в разы быстрее `GradientBoostingRegressor` при том же качестве.

Построение деревьев и бустинга по шагам — в интерактивной демонстрации А. Рогожникова
[Gradient Boosting explained](https://arogozhnikov.github.io/2016/06/24/gradient_boosting_explained.html).
На двумерной задаче регрессии видно, как дерево делит плоскость признаков на области,
а в бустинге — сумма построенных деревьев, ее остаток и следующее дерево, обученное
на остатке. Глубину деревьев и их число меняют ползунками, темп обучения $\eta = 0.3$.

Следующее занятие — стекинг (stacking): ответы нескольких разных моделей
объединяет еще одна модель, обученная на их предсказаниях.